# 🦴 Bone_Age_Predict — Colab Demo

Run the open-source YOLOv5 + RUS-CHN bone-age pipeline on the repository sample image.

> **Research and educational use only.** This notebook is not a medical device and must not be used as an independent clinical diagnosis.


In [ ]:
!git clone --depth 1 https://github.com/Seazer-x/Bone_Age_Predict.git
%cd Bone_Age_Predict


In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'packaging>=24.1', 'opencv-python-headless==4.10.0.84', 'Pillow==10.4.0',
    'matplotlib==3.9.2', 'pandas==2.2.3', 'PyYAML==6.0.2',
    'scipy==1.14.1', 'seaborn==0.13.2', 'ipython==8.27.0',
    'psutil==6.0.0', 'thop==0.1.1.post2209072238', 'tqdm==4.66.5'
])

from packaging.version import Version
import torch
if Version(torch.__version__.split('+')[0]) < Version('2.8.0'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.8.0', 'torchvision==0.23.0'])
    print('PyTorch was upgraded. Restart the runtime, then continue from this cell.')
else:
    print('PyTorch:', torch.__version__)


In [ ]:
import os
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from IPython.display import display

os.environ.setdefault('TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD', '1')

from bone_age.bone_age import Bone_Age

MODEL_NAMES = ['Radius', 'Ulna', 'MCPFirst', 'MCP', 'PIP', 'PIPFirst', 'MIP', 'DIP', 'DIPFirst']
weights = [Path('bone_age') / name / 'best.pt' for name in MODEL_NAMES]
detector = Path('bone_age/bone_age.pt')
device = '0' if torch.cuda.is_available() else 'cpu'

predictor = Bone_Age(weights, MODEL_NAMES, device=device)
print('Device:', device)


In [ ]:
sample = Path('images/1778216012401.jpg')
image = Image.open(sample).convert('RGB')
display(image)

report, success = predictor.run(
    detector,
    'boy',
    np.asarray(image),
    conf_thres=0.40,
    iou_thres=0.45,
)
print(report)
assert success, report


## Evaluate a labeled dataset

For reproducible MAE/RMSE evaluation, prepare a CSV with `image,sex,age_years` and run:

```bash
python evaluation/evaluate_dataset.py --manifest /content/manifest.csv --device 0
```

See [`evaluation/README.md`](https://github.com/Seazer-x/Bone_Age_Predict/blob/main/evaluation/README.md) for the protocol.
